# Qwen3.5-0.8B × TinyCeNN Memory Fusion — Sequential Acceptance

This version uses the same **live training style** as `Qwen3_5_0_8B_CeNN_Integrated_Memory_V2_Colab.ipynb`: the trainer is run with `subprocess.Popen`, every line is streamed immediately, the same output is written to Google Drive, and a live dashboard shows anchor / round / step / acceptance metrics.

Qwen3.5 native Gated DeltaNet (`linear_attention`) layers stay unchanged. TinyCeNN Memory Fusion is trained only on full-attention anchors **3, 7, 11, 15, 19, 23**.


In [ ]:
import os,sys,json,subprocess,tempfile,shutil
from pathlib import Path
assert subprocess.run(['nvidia-smi'],check=False).returncode==0,'Enable a GPU runtime'
REPO=Path(tempfile.mkdtemp(prefix='qwen35-memoryfusion-'))/'TinyCeNN-LM'
subprocess.run(['git','clone','--quiet','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO)],check=True)
subprocess.run(['git','fetch','origin','main'],cwd=REPO,check=True)
subprocess.run(['git','reset','--hard','origin/main'],cwd=REPO,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==5.17.0','datasets','huggingface_hub','accelerate','safetensors','pytest','pandas','matplotlib'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),'--no-deps'],check=True)
os.environ['PYTHONPATH']=os.pathsep.join([str(REPO),str(REPO/'src')]); sys.path[:0]=[str(REPO),str(REPO/'src')]
import torch
SOURCE=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
print('Source:',SOURCE); print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


In [ ]:
from google.colab import drive
from huggingface_hub import HfApi
from transformers import AutoConfig
drive.mount('/content/drive')
BASE_MODEL='Qwen/Qwen3.5-0.8B'; MODEL_REVISION=HfApi().model_info(BASE_MODEL).sha
PROFILE='balanced' # @param ['smoke','balanced','extended']
RESET_PROGRESS=False # @param {type:'boolean'}
PUBLISH_TO_HF=True # @param {type:'boolean'}
PROFILES={'smoke':dict(context=64,probe=64,steps=75,check=25,rounds=1),'balanced':dict(context=128,probe=128,steps=300,check=25,rounds=4),'extended':dict(context=256,probe=256,steps=450,check=25,rounds=6)}
RUN=PROFILES[PROFILE]; FEATURE_DIM=32; MEMORY_RANK=64; SEED=73
CONTEXT=RUN['context']; PROBE_CONTEXT=RUN['probe']; MAX_LAYER_STEPS=RUN['steps']; CHECK_EVERY=RUN['check']; MAX_ROUNDS_PER_RUN=RUN['rounds']
HF_MODEL_REPO='vtava/Qwen3.5-0.8B-MemoryFusion'; HF_PRIVATE=False
OUTPUT_DIR=Path('/content/drive/MyDrive/TinyCeNN-LM/qwen35-0.8b-memory-fusion-sequential-r64')
if RESET_PROGRESS and OUTPUT_DIR.exists(): shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
cfg=AutoConfig.from_pretrained(BASE_MODEL,revision=MODEL_REVISION).get_text_config(decoder=True)
TARGET_LAYERS=[i for i,t in enumerate(cfg.layer_types) if t=='full_attention']; LINEAR_LAYERS=[i for i,t in enumerate(cfg.layer_types) if t=='linear_attention']
assert TARGET_LAYERS==[3,7,11,15,19,23],TARGET_LAYERS
print(json.dumps({'profile':PROFILE,'revision':MODEL_REVISION,'targets':TARGET_LAYERS,'native_linear':len(LINEAR_LAYERS),'context':CONTEXT,'probe_context':PROBE_CONTEXT,'max_steps':MAX_LAYER_STEPS,'rounds_this_run':MAX_ROUNDS_PER_RUN,'output':str(OUTPUT_DIR),'hf_repo':HF_MODEL_REPO},indent=2))


In [ ]:
from huggingface_hub import HfApi,login,notebook_login
from google.colab import userdata
try: token=userdata.get('HF_TOKEN')
except Exception: token=None
if token: os.environ['HF_TOKEN']=token; login(token=token,add_to_git_credential=False)
else: notebook_login()
print('✅ HF user',HfApi().whoami().get('name'))


In [ ]:
env=dict(os.environ,CUDA_VISIBLE_DEVICES='',OMP_NUM_THREADS='1',MKL_NUM_THREADS='1'); env['TINYCENN_PARENT_BACKUP_ACTIVE']='1'
r=subprocess.run([sys.executable,'-m','pytest','-q','tests/test_qwen3_5_memory_fusion.py'],cwd=REPO,env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(r.stdout)
if r.returncode: raise RuntimeError(f'preflight failed: {r.returncode}')
print('✅ preflight passed')


## Saved progress before training

This dashboard is rebuilt from the checkpoint/status files in Drive, so it survives Colab disconnects.


In [ ]:
from tinycenn_lm.qwen35_memory_fusion_colab import show_progress,run_live_training,DEFAULT_USER_CHATS
PROGRESS_BEFORE=show_progress(OUTPUT_DIR,TARGET_LAYERS)


## Train / resume — streamed like Integrated Memory V2

The next cell prints every training line immediately **and** updates the live status panel with the current anchor, round, step, NMSE, cosine and ΔNLL. The full stream is saved as `last_colab_run.log` in Drive.


In [ ]:
cmd=[sys.executable,'-u',str(REPO/'scripts'/'train_qwen35_memory_fusion_sequential.py'),'--base-model',BASE_MODEL,'--model-revision',MODEL_REVISION,'--output-dir',str(OUTPUT_DIR),'--feature-dim',str(FEATURE_DIM),'--memory-rank',str(MEMORY_RANK),'--context-length',str(CONTEXT),'--probe-context',str(PROBE_CONTEXT),'--seed',str(SEED),'--min-layer-steps','50','--max-layer-steps',str(MAX_LAYER_STEPS),'--check-every',str(CHECK_EVERY),'--layer-lr','0.0002','--teacher-alpha-start','0.9','--teacher-alpha-end','0.0','--accept-nmse','0.2','--accept-cosine','0.9','--accept-incremental-delta-nll','0.015','--accept-cumulative-delta-nll','0.05','--max-runtime-minutes','240','--resume','--strict-acceptance']
run_env=dict(os.environ); run_env['SEQUENTIAL_MAX_ROUNDS_PER_RUN']=str(MAX_ROUNDS_PER_RUN)
return_code=run_live_training(cmd=cmd,repo=REPO,output_dir=OUTPUT_DIR,target_layers=TARGET_LAYERS,max_step=MAX_LAYER_STEPS,env=run_env)
if return_code: raise RuntimeError(f'trainer failed with exit code {return_code}; see {OUTPUT_DIR/"last_colab_run.log"}')
print('✅ trainer returned normally; needs_more_training is a resumable scientific status, not a crash')
print('Updated saved progress:'); PROGRESS_AFTER=show_progress(OUTPUT_DIR,TARGET_LAYERS)


In [ ]:
for name in ('sequential_run_status.json','sequential_progress.json','sequential_in_progress.json','sequential_training_report.json'):
    p=OUTPUT_DIR/name
    if p.exists(): print('\n###',name); print(p.read_text(encoding='utf-8'))


## Real user-chat comparison: original Qwen3.5 vs accepted Memory Fusion

Only accepted anchors are loaded; the current unaccepted layer is excluded. Both models use Qwen's chat template and deterministic decoding.


In [ ]:
import torch
from transformers import AutoTokenizer,Qwen3_5ForCausalLM
from tinycenn_lm.qwen3_5_memory_fusion import Qwen35MemoryFusionConfig,replace_attention_layers,load_selected_attention_state,structural_summary
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); DTYPE=torch.bfloat16 if DEVICE.type=='cuda' and torch.cuda.is_bf16_supported() else (torch.float16 if DEVICE.type=='cuda' else torch.float32)
pp=OUTPUT_DIR/'sequential_progress.pt'; payload=torch.load(pp,map_location='cpu',weights_only=False) if pp.exists() else None
accepted=[int(x) for x in payload.get('accepted_layers',[])] if payload else []; mf_cfg=Qwen35MemoryFusionConfig.from_dict(payload['config']) if payload else Qwen35MemoryFusionConfig(feature_dim=FEATURE_DIM,memory_rank=MEMORY_RANK)
tok=AutoTokenizer.from_pretrained(BASE_MODEL,revision=MODEL_REVISION)
original=Qwen3_5ForCausalLM.from_pretrained(BASE_MODEL,revision=MODEL_REVISION,dtype=DTYPE,attn_implementation='sdpa',low_cpu_mem_usage=True).to(DEVICE).eval()
adapted=Qwen3_5ForCausalLM.from_pretrained(BASE_MODEL,revision=MODEL_REVISION,dtype=DTYPE,attn_implementation='sdpa',low_cpu_mem_usage=True).to(DEVICE).eval()
if accepted: replace_attention_layers(adapted,mf_cfg,accepted); load_selected_attention_state(adapted,payload['attention_state'],accepted)
original.config.use_cache=False; adapted.config.use_cache=False
print('Accepted anchors:',accepted); print(json.dumps(structural_summary(adapted),indent=2))
@torch.no_grad()
def chat(model,msgs):
    text=tok.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True); batch=tok(text,return_tensors='pt').to(DEVICE)
    out=model.generate(**batch,max_new_tokens=128,do_sample=False,use_cache=False,pad_token_id=tok.eos_token_id)
    return tok.decode(out[0,batch['input_ids'].shape[1]:],skip_special_tokens=True).strip()
examples=[]
for i,case in enumerate(DEFAULT_USER_CHATS,1):
    old,new=chat(original,case['messages']),chat(adapted,case['messages']); print('\n'+'='*100); print(f"CHAT {i}: {case['name']}"); print('USER:',case['messages'][-1]['content']); print('\nORIGINAL QWEN3.5:\n',old); print('\nMEMORY FUSION:\n',new); examples.append({'name':case['name'],'messages':case['messages'],'original':old,'memory_fusion':new,'accepted_layers':accepted})
(OUTPUT_DIR/'chat_examples.json').write_text(json.dumps(examples,indent=2,ensure_ascii=False),encoding='utf-8'); print('✅ saved',OUTPUT_DIR/'chat_examples.json')


## Publish accepted model state + resumable current state to Hugging Face


In [ ]:
from tinycenn_lm.qwen3_5_memory_fusion import save_adapter
PACKAGE=Path('/content/qwen35-memory-fusion-hf')
if PACKAGE.exists(): shutil.rmtree(PACKAGE)
PACKAGE.mkdir(parents=True); (PACKAGE/'training').mkdir()
if accepted: save_adapter(adapted,PACKAGE,config=mf_cfg,base_model=BASE_MODEL,accepted_layers=accepted,metadata={'base_revision':MODEL_REVISION,'source_repo':'vtavakkoli/TinyCeNN-LM','source_commit':SOURCE})
for name in ('sequential_run_status.json','sequential_progress.json','sequential_in_progress.json','sequential_training_report.json','chat_examples.json','last_colab_run.log'):
    src=OUTPUT_DIR/name
    if src.exists(): shutil.copy2(src,PACKAGE/name)
for name in ('sequential_in_progress.pt','sequential_progress.pt','qwen35_memory_fusion_full_state.pt'):
    src=OUTPUT_DIR/name
    if src.exists(): shutil.copy2(src,PACKAGE/'training'/name)
card=f'''---
library_name: transformers
base_model: {BASE_MODEL}
license: apache-2.0
tags: [qwen3.5, tinycenn, memory-fusion, recurrent-attention]
---
# Qwen3.5-0.8B × TinyCeNN Memory Fusion

Experimental accepted-only Memory Fusion adapter for `{BASE_MODEL}`. Native Qwen3.5 Gated DeltaNet layers remain unchanged.

**Target anchors:** `[3, 7, 11, 15, 19, 23]`  
**Accepted anchors:** `{accepted}`  
**Base revision:** `{MODEL_REVISION}`  
**Source commit:** `{SOURCE}`

Acceptance gates: NMSE ≤ 0.20, cosine ≥ 0.90, incremental ΔNLL ≤ 0.015, cumulative ΔNLL ≤ 0.05. `chat_examples.json` contains deterministic original-vs-Memory-Fusion chat tests. The unaccepted current layer, when present, is stored only under `training/`.
'''
(PACKAGE/'README.md').write_text(card,encoding='utf-8')
api=HfApi()
if PUBLISH_TO_HF:
    api.create_repo(HF_MODEL_REPO,repo_type='model',private=HF_PRIVATE,exist_ok=True); api.upload_folder(repo_id=HF_MODEL_REPO,repo_type='model',folder_path=str(PACKAGE),commit_message=f'Update Qwen3.5 Memory Fusion accepted={accepted}'); print('✅ Published: https://huggingface.co/'+HF_MODEL_REPO)
else: print('Publication disabled; package ready at',PACKAGE)
